In [1]:
import pandas as pd

# =====================================================
# OVERLAP PERIOD FROM YOUR NQ_MERGED
# =====================================================

START_DATE = "2025-02-21 00:00:00+00:00"
END_DATE   = "2026-02-20 21:59:00+00:00"

print("Loading Databento CSV...")

df = pd.read_csv(
    "databento.csv",
    usecols=[
        "ts_event",
        "open",
        "high",
        "low",
        "close",
        "volume",
        "symbol"
    ]
)

print(f"Loaded {len(df):,} rows")

# =====================================================
# TIMESTAMP CONVERSION
# =====================================================

print("Converting timestamps...")

df["datetime"] = pd.to_datetime(
    df["ts_event"],
    utc=True
)

# =====================================================
# FILTER TO RELEVANT DATE RANGE
# =====================================================

start = pd.Timestamp(START_DATE)
end = pd.Timestamp(END_DATE)

df = df[
    (df["datetime"] >= start)
    &
    (df["datetime"] <= end)
]

print(f"Rows after date filter: {len(df):,}")

# =====================================================
# REMOVE SPREADS
# =====================================================

df = df[
    ~df["symbol"].str.contains("-", regex=False)
]

print(f"Rows after removing spreads: {len(df):,}")

print("\nContracts remaining:")

print(
    df["symbol"]
    .value_counts()
    .sort_index()
)

# =====================================================
# BUILD CONTINUOUS SERIES
# =====================================================

print("\nBuilding continuous contract...")

# highest volume contract wins
df = df.sort_values(
    ["datetime", "volume"],
    ascending=[True, False]
)

continuous = (
    df.groupby("datetime", as_index=False)
      .first()
)

continuous = continuous[
    [
        "datetime",
        "symbol",
        "open",
        "high",
        "low",
        "close",
        "volume"
    ]
]

print(f"Continuous rows: {len(continuous):,}")

# =====================================================
# SAVE
# =====================================================

continuous.to_csv(
    "databento_continuous.csv",
    index=False
)

print("\nSaved databento_continuous.csv")

print("\nContract usage:")

print(
    continuous["symbol"]
    .value_counts()
    .sort_index()
)

Loading Databento CSV...
Loaded 7,136,029 rows
Converting timestamps...
Rows after date filter: 518,527
Rows after removing spreads: 456,315

Contracts remaining:
symbol
NQH5     28188
NQH6     84805
NQM5    105357
NQM6     11681
NQU5    115745
NQU6       107
NQZ5    110428
NQZ6         2
NQZ7         2
Name: count, dtype: int64

Building continuous contract...
Continuous rows: 352,957

Saved databento_continuous.csv

Contract usage:
symbol
NQH5    23449
NQH6    64970
NQM5    87011
NQU5    89414
NQZ5    88113
Name: count, dtype: int64
